In [12]:
wg_sets = [
    {
        "description": "First spatial lag, group concentric rings",
        "variables": [
            "y",
            "k",
            "l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "All combinations of second-order group spatial lags",
        "variables": [
            "wg1_k", "wg1_l",
            "wg2_k", "wg2_l",
            "wg3_k", "wg3_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "Third and 4th order group lags for wg1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8"
        }
    },
    {
        "description": "Local and global FE transforms for w2g1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-wg1)": {
                "i_minus": True,
                "value": "pc8"
            },
            "(i-vg1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "pc8"
            }
        }
    }
]

wd_sets = [
    {
        "description": "Distance-weighted spatial lags",
        "variables": [
            "y",
            "k",
            "l"
        ],
        "type": "n",
        "transforms": [
            "d1",
            "d2",
            "d3"
        ]
    },
    {
        "description": "Higher-index d1 lags",
        "variables": [
            "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": [
            "d1"
        ]
    },
    {
        "description": "Higher-index d2 lags",
        "variables": [
            "wd2_k", "wd2_l"
        ],
        "type": "n",
        "transforms": [
            "d2"
        ]
    },
    {
        "description": "Higher-index d3 lags",
        "variables": [
            "wd3_k", "wd3_l"
        ],
        "type": "n",
        "transforms": [
            "d3"
        ]
    },
    {
        "description": "Local transforms for w2d1",
        "variables": [
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-wd1)": {
                "i_minus": True,
                "value": "d1"
            }
        }
    },
    {
        "description": "Global transforms for w2d1",
        "variables": [
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-vd1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "d1"
            }
        }
    }
]

In [13]:
import re
from site import PREFIXES

# Parses and generates the output column name based on transformation rules.
# Automatically increments indices (e.g., w2g1 -> w3g1) if the transformations match.
def get_out_name(var_name: str, transform_name: str) -> str:
    if "_" in var_name:
        parts = var_name.split("_", 1)
        prefix = parts[0]
        suffix = parts[1]
        
        # Check if the last transformation matches the new one exactly
        match = re.search(r'w(\d+)?' + re.escape(transform_name) + r'$', prefix)

        if transform_name.startswith("(") and transform_name.endswith(")"):
            new_prefix = f"{transform_name}{prefix}"
        elif match:
            num = int(match.group(1)) if match.group(1) else 1
            new_prefix = prefix[:match.start()] + f"w{num + 1}{transform_name}"
        else:
            new_prefix = f"w{transform_name}{prefix}"
            
        return f"{new_prefix}_{suffix}"
    else:
        # Base variable renaming (e.g., 'y' -> 'wg1_y')
        
        if transform_name.startswith("(") and transform_name.endswith(")"):
            return f"{transform_name}{var_name}"
        else:
            return f"w{transform_name}_{var_name}"

transform_tree = { "g": [], "n": [] }
mapping_dict = { "g": {}, "n": {} }
for w_set in wg_sets + wd_sets:
    variables = w_set["variables"]
    transforms = w_set["transforms"]
    w_type = w_set["type"]
    
    # Normalize inputs to dictionaries for uniform loop processing
    if isinstance(transforms, list):
        transforms = {t: t for t in transforms}
    if isinstance(variables, list):
        variables = {v: v for v in variables}

    for new_base, current_col in variables.items():
        for t_name, t_val in transforms.items():
            
            # Parse output string
            out_col = get_out_name(new_base, t_name)
            
            # Parse transformation attributes
            w_col: str                      = t_val
            i_minus: bool                   = False
            leave_one_out: bool             = True
            is_hybrid: bool                 = False
            inner_cols: list[str] | None    = None
            if isinstance(t_val, dict):                
                weight_val = t_val.get("value")
                if type(weight_val) is not str:
                    raise ValueError(f"Expected string for weight value, got {type(weight_val)}: {weight_val}")
                w_col = weight_val
                i_minus = t_val.get("i_minus", False)
                inner_cols = t_val.get("inner", None)
                leave_one_out = t_val.get("leave_one_out", True)
                is_hybrid = "leave_one_out" in t_val  # Triggers overlapping individual graph cluster logic
            
            # Dispatch to appropriate mathematical pipeline 
            if w_type == "g":
                transform_tree["g"].append({
                    "input_col": current_col,
                    "group_col": w_col,
                    "out_col": out_col,
                    "inner_cols": inner_cols,
                    "leave_one_out": leave_one_out,
                    "i_minus": i_minus,
                })
                mapping_dict["g"][out_col] = current_col
            elif w_type == "n":
                transform_tree["n"].append({
                    "input_col": current_col,
                    "weight_col": w_col,
                    "out_col": out_col,
                    "leave_one_out": leave_one_out,
                    "i_minus": i_minus,
                })
                mapping_dict["n"][out_col] = current_col

# Add depth property to each entry in transformation tree
# Based on following the mapping_dict backwards from each out_col to its input_col, counting the number of steps until reaching a base variable (not in mapping_dict).
for w_type in transform_tree:
    for transform in transform_tree[w_type]:
        depth = 1
        current_col = transform["input_col"]
        while current_col in mapping_dict[w_type]:
            current_col = mapping_dict[w_type][current_col]
            depth += 1
        transform["depth"] = depth

print("Transformation tree built successfully.")
# Output transformation tree as json for inspection
import json
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
outfile = dirs.tmp_dir / "transform_tree.json"
with open(outfile, "w") as f:
    json.dump(transform_tree, f, indent=4)

# For each depth level, list the variable-transformation pairs that will take place at that depth.
for w_type in transform_tree:
    max_depth = max(t["depth"] for t in transform_tree[w_type]) if transform_tree[w_type] else 0
    print(f"\n{w_type}-type transformations by depth:")
    for depth in range(1, max_depth + 1):
        depth_transforms = [t for t in transform_tree[w_type] if t["depth"] == depth]
        print(f"Depth {depth}: {[f'{t['input_col']} -> {t['out_col']}' for t in depth_transforms]}")

Transformation tree built successfully.

g-type transformations by depth:
Depth 1: ['y -> wg1_y', 'y -> wg2_y', 'y -> wg3_y', 'k -> wg1_k', 'k -> wg2_k', 'k -> wg3_k', 'l -> wg1_l', 'l -> wg2_l', 'l -> wg3_l']
Depth 2: ['wg1_k -> w2g1_k', 'wg1_k -> wg2wg1_k', 'wg1_k -> wg3wg1_k', 'wg1_l -> w2g1_l', 'wg1_l -> wg2wg1_l', 'wg1_l -> wg3wg1_l', 'wg2_k -> wg1wg2_k', 'wg2_k -> w2g2_k', 'wg2_k -> wg3wg2_k', 'wg2_l -> wg1wg2_l', 'wg2_l -> w2g2_l', 'wg2_l -> wg3wg2_l', 'wg3_k -> wg1wg3_k', 'wg3_k -> wg2wg3_k', 'wg3_k -> w2g3_k', 'wg3_l -> wg1wg3_l', 'wg3_l -> wg2wg3_l', 'wg3_l -> w2g3_l']
Depth 3: ['w2g1_k -> w3g1_k', 'w2g1_l -> w3g1_l', 'w2g1_k -> (i-wg1)w2g1_k', 'w2g1_k -> (i-vg1)w2g1_k', 'w2g1_l -> (i-wg1)w2g1_l', 'w2g1_l -> (i-vg1)w2g1_l']
Depth 4: ['w3g1_k -> w4g1_k', 'w3g1_l -> w4g1_l', 'w3g1_k -> (i-wg1)w3g1_k', 'w3g1_k -> (i-vg1)w3g1_k', 'w3g1_l -> (i-wg1)w3g1_l', 'w3g1_l -> (i-vg1)w3g1_l']

n-type transformations by depth:
Depth 1: ['y -> wd1_y', 'y -> wd2_y', 'y -> wd3_y', 'k -> wd1_k'

In [ ]:
import ibis
from ibis import _
import pandas as pd
import itertools
from utils.f_0_dirs import get_data_dirs

# Batches multiple group-based spatial lags and processes them in a single Ibis mutation.
# Calculates mutually exclusive donut holes (Inclusion-Exclusion) and (I-W) fixed effects.
def apply_group_W(t_panel: ibis.Table, transforms: list[dict]) -> ibis.Table:

    new_cols = {}
    
    for trans_def in transforms:
        input_col = trans_def["input_col"]
        group_col = trans_def["group_col"]
        out_col = trans_def["out_col"]
        inner_cols = trans_def.get("inner_cols")
        leave_one_out = trans_def.get("leave_one_out", True)
        i_minus = trans_def.get("i_minus", False)

        # 1. Base group sum and count
        sum_col = _[input_col].sum().over(group_by=[_[group_col], _.year])
        count_col = _[input_col].count().over(group_by=[_[group_col], _.year])
        
        # 2. Subtract inner groups if specified (Inclusion-Exclusion Principle)
        if not inner_cols:
            # 3. Standard group (no donut hole)
            if leave_one_out:
                numerator = sum_col - ibis.coalesce(_[input_col], 0)
                denominator = count_col - 1
            else:
                numerator = sum_col
                denominator = count_col
        else:
            sub_sum = None
            sub_count = None
            
            for r in range(1, len(inner_cols) + 1):
                sign = 1 if r % 2 != 0 else -1
                for combo in itertools.combinations(inner_cols, r):
                    part_cols = [_[group_col], _.year] + [_[c] for c in combo]
                    
                    term_sum = _[input_col].sum().over(group_by=part_cols)
                    term_count = _[input_col].count().over(group_by=part_cols)
                    
                    if sub_sum is None:
                        sub_sum = ibis.coalesce(term_sum, 0)
                        sub_count = ibis.coalesce(term_count, 0)
                    else:
                        sub_sum = sub_sum + (ibis.coalesce(term_sum, 0) * sign)         # type: ignore
                        sub_count = sub_count + (ibis.coalesce(term_count, 0) * sign)   # type: ignore
            
            base_sum = sum_col - sub_sum
            base_count = count_col - sub_count
            
            if leave_one_out:
                numerator = base_sum
                denominator = base_count
            else:
                numerator = base_sum + ibis.coalesce(_[input_col], 0)
                denominator = base_count + 1                
            
        # 4. Row-normalize and apply (I - W) logic
        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        new_cols[out_col] = _[input_col] - spatial_lag if i_minus else spatial_lag

    # Apply all transformations at this depth in one deferred AST branch
    return t_panel.mutate(**new_cols)

# Batches multiple distance-decay and overlapping individual network group lags.
# Executes a single structural join per depth level.
def apply_network_W(t_panel: ibis.Table, t_distance: ibis.Table, transforms: list[dict]) -> ibis.Table:

    input_cols = list(set(t["input_col"] for t in transforms))

    # 1. Build deferred selections to isolate target and peer data dynamically
    target_selects = {"target_firm": _.registered_number, "target_year": _.year}
    peer_selects = {"peer_firm": _.registered_number, "peer_year": _.year}

    for col in input_cols:
        target_selects[f"target_{col}"] = _[col]
        peer_selects[f"peer_{col}"] = _[col]

    t_target = t_panel.select(**target_selects)
    t_peer = t_panel.select(**peer_selects)

    # 2. Single Master Join (Chain safely evaluates `_` as the growing left-side table)
    t_joined = (
        t_distance
        .inner_join(t_target, _.firm_i == t_target.target_firm)                                     # type: ignore
        .inner_join(t_peer, (_.firm_j == t_peer.peer_firm) & (_.target_year == t_peer.peer_year))   # type: ignore
    )

    # 3. Build deferred aggregation dictionary for all transforms concurrently
    agg_exprs = {}
    for trans_def in transforms:
        in_col = trans_def["input_col"]
        w_col = trans_def["weight_col"]
        out_col = trans_def["out_col"]
        leave_one_out = trans_def.get("leave_one_out", True)
        i_minus = trans_def.get("i_minus", False)
        
        is_hybrid = not leave_one_out 

        peer_val = _[f"peer_{in_col}"]
        target_val = _[f"target_{in_col}"]
        weight_val = _[w_col]

        if is_hybrid:
            # Overlapping individual group
            peer_sum = peer_val.sum()
            peer_count = peer_val.count()
            
            numerator = peer_sum + ibis.coalesce(target_val.first(), 0)
            denominator = peer_count + 1
        else:
            # Continuous network distance decay
            weighted_val = peer_val * weight_val
            numerator = weighted_val.sum()
            denominator = ibis.ifelse(peer_val.notnull(), weight_val, ibis.null()).sum()        # type: ignore

        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        agg_exprs[out_col] = target_val.first() - spatial_lag if i_minus else spatial_lag

    # 4. Perform the massive single aggregation
    t_distance_weighted = (
        t_joined
        .group_by([_.firm_i, _.target_year])
        .aggregate(**agg_exprs)
    )

    # 5. Join back and drop redundant identifiers cleanly
    t_panel_mutated = (
        t_panel
        .left_join(
            t_distance_weighted,
            (_.registered_number == t_distance_weighted.firm_i) & (_.year == t_distance_weighted.target_year)   # type: ignore
        )
        .drop("firm_i", "target_year")
    )
    
    return t_panel_mutated


# ==========================================
# Schema Iteration & Table Management
# ==========================================

# Assuming 'transform_tree' contains your parsed JSON dict
# transform_tree = json.loads(json_string)

panel_name = "working_yearly"
fixed_name = "working_fixed"
distance_name = "working_distance_ttwa_km"

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

table_panel = (
    con.table(panel_name)
    .select("registered_number", "year", "gva1", "total_assets", "employees")
    .rename({"gva": "gva1"})
    .distinct(on=["registered_number", "year"])
    .filter(
        (_["gva"].notnull())            & (_["gva"] > 0) &
        (_["total_assets"].notnull())   & (_["total_assets"] > 0) &
        (_["employees"].notnull())      & (_["employees"] > 0)
    )
    .mutate(
        y = _['gva'].log(),
        k = _['total_assets'].log(),
        l = _['employees'].log()
    )
)

table_fixed = (
    con.table(fixed_name)
    .select("registered_number", "pc8", "pc4", "ttwa")
    .distinct(on="registered_number")
)

# Constrain the universe of distances to only valid firms mapped in the dataset
table_uniques = (
    table_panel
    .inner_join(table_fixed, "registered_number")
    .distinct(on="registered_number")
    .select("registered_number")
)

table_distance = (
    con.table(distance_name)
    .inner_join(table_uniques, _.firm_i == table_uniques.registered_number)
    .mutate(
        d1 = 1 / (_.distance_meters + 1),
        d2 = 1 / (_.distance_meters + 1) ** 2,
        d3 = (-_.distance_meters / 1000).exp()
    )
)

should_run = 'both' # Options: 'g', 'n', 'both'

# 1m14s run
if should_run == 'g' or should_run == 'both':
    # Run groups
    t_current_g = table_panel.left_join(table_fixed, "registered_number")
    max_depth_g = max(t.get("depth", 1) for t in transform_tree.get("g", []))
    print(f"\nBeginning group transformations, {len(transform_tree.get('g', []))} required across depth {max_depth_g}.")
    for d in range(1, max_depth_g + 1):
        g_transforms = [t for t in transform_tree.get("g", []) if t.get("depth") == d]
        print(f"--- applying depth {d} with {len(g_transforms)} transformations, {len(t_current_g.columns)} columns...")
        if not g_transforms:
            print(f"No group transformations found at depth {d}.")
            continue
        t_current_g = apply_group_W(t_current_g, g_transforms)
    t_final_g = (
        t_current_g
        .drop("registered_number_right", "pc8", "pc4", "ttwa")
    )
    con.create_table("working_yearly_g", t_final_g, overwrite=True)
    print(con.table("working_yearly_g").sample(0.001).execute())

# 1m24s for depth: 1
# 10 minute run, out of memory
# 1m30s per depth. Should be around 7 minutes total
if should_run == 'n' or should_run == 'both':
    # Run networks
    time_start = pd.Timestamp.now()
    t_start_n = table_panel.left_join(table_fixed, "registered_number")
    max_depth_n = max(t.get("depth", 1) for t in transform_tree.get("n", []))
    print(f"\nBeginning network transformations writing to db at each step, {len(transform_tree.get('n', []))} required across depth {max_depth_n}.")
    write_name = "working_yearly_n"
    
    for d in range(1, max_depth_n + 1):
        try:
            t_imported_n = con.table(write_name)
            t_imported_rows = t_imported_n.count().execute()
            t_imported_columns = t_imported_n.columns
            if t_imported_rows == 0 or len(t_imported_columns) < 5:
                raise ValueError(f"Imported table {write_name} is empty or has insufficient columns.")
            t_current_n = t_imported_n
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- got table from {write_name} with shape ({t_imported_rows}, {len(t_imported_columns)})")
        except Exception as e:
            t_current_n = t_start_n
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- starting from t_start_n with shape ({t_start_n.count().execute()}, {len(t_start_n.columns)}) for depth {d} due to: {e}")

        n_transforms = [t for t in transform_tree.get("n", []) if t.get("depth") == d]
        time_check = (pd.Timestamp.now() - time_start).total_seconds()
        print(f"[{time_check:.1f}s] --- applying depth {d} with {len(n_transforms)} transformations, {len(t_current_n.columns)} columns...")
        if not n_transforms:
            print(f"No network transformations found at depth {d}.")
            continue
        t_out_n = apply_network_W(t_current_n, table_distance, n_transforms)
        for col in ["registered_number_right", "pc8", "pc4", "ttwa"]:
            if col in t_out_n.columns:
                t_out_n = t_out_n.drop(col)
        con.create_table("working_yearly_n", t_out_n, overwrite=True)

    print(con.table("working_yearly_n").sample(0.001).execute())



Beginning group transformations, 39 required across depth 4.
--- applying depth 1 with 9 transformations, 12 columns...
--- applying depth 2 with 18 transformations, 21 columns...
--- applying depth 3 with 6 transformations, 39 columns...
--- applying depth 4 with 6 transformations, 45 columns...
     registered_number  year           gva   total_assets  employees  \
0             00449706  2010  17874.039697   57165.259053        254   
1             02072364  2010  11559.711633   20196.504546        165   
2             03981392  2019  23909.596338   39380.758509        682   
3             01494909  2024   4164.000000    7293.000000        197   
4             05935923  2013  28688.529606   68497.743119        511   
...                ...   ...           ...            ...        ...   
1090          02107821  2021  30773.002217  187844.195506        240   
1091          04301304  2006   6522.014867   14200.364589        117   
1092          03897075  2022   6054.759022    3104.80

In [ ]:
# %%script drop table
# Drop table "working_yearly_n" if it exists
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

drop = False
if drop:
    if con.table("working_yearly_n").columns:
        con.drop_table("working_yearly_n")
        print("Dropped table 'working_yearly_n' after processing.")

rename = False
if rename:
    table_name = "working_yearly_g"
    in_table = con.table(table_name)
    out_table = (
        in_table
        # .drop('pc8', 'pc4', 'ttwa', 'registered_number_right')
        .drop('registered_number_right')
        .rename({
            # '(i-wd1)w2d1_k': 'w(i-wd1)w2d1_k',
            # '(i-wd1)w2d1_l': 'w(i-wd1)w2d1_l',
            # '(i-vd1)w2d1_k': 'w(i-vd1)w2d1_k',
            # '(i-vd1)w2d1_l': 'w(i-vd1)w2d1_l',
            # '(i-wd1)w3d1_k': 'w(i-wd1)w3d1_k',
            # '(i-wd1)w3d1_l': 'w(i-wd1)w3d1_l',
            # '(i-vd1)w3d1_k': 'w(i-vd1)w3d1_k',
            # '(i-vd1)w3d1_l': 'w(i-vd1)w3d1_l'

            '(i-wg1)w2g1_k': 'w(i-wg1)w2g1_k',
            '(i-wg1)w2g1_l': 'w(i-wg1)w2g1_l',
            '(i-vg1)w2g1_k': 'w(i-vg1)w2g1_k',
            '(i-vg1)w2g1_l': 'w(i-vg1)w2g1_l',
            '(i-wg1)w3g1_k': 'w(i-wg1)w3g1_k',
            '(i-wg1)w3g1_l': 'w(i-wg1)w3g1_l',
            '(i-vg1)w3g1_k': 'w(i-vg1)w3g1_k',
            '(i-vg1)w3g1_l': 'w(i-vg1)w3g1_l'
        })
    )
    print(out_table.columns)
    con.create_table(table_name, out_table, overwrite=True)

# 2. Read and sample to verify

In [19]:
t_distance = con.table("working_yearly_g")
print(t_distance.limit(10).execute())
out_file = dirs.tmp_dir / "w_transforms_sample_d.xlsx"
t_distance.sample(0.001).execute().to_excel(out_file, index=False)
print(f"Exported check to {out_file}")

  registered_number  year          gva  total_assets  employees         y  \
0          04464220  2009  5062.719843   5996.177443        134  8.529659   
1          11425513  2020  2563.924112   9709.478507         70  7.849294   
2          02132170  2011   927.324632    249.451760         13  6.832304   
3          00598760  2008  2923.641856   2940.254888         54  7.980585   
4          06228171  2011    97.563919    113.939902         10  4.580508   
5          02994954  2017   353.990178    921.934183         23  5.869269   
6          01481734  2017  4477.377772   9023.323605         31  8.406793   
7          01481734  2022  2653.136770   6582.900621         32  7.883498   
8          05938886  2022  6673.480946  12791.429647        113  8.805897   
9          02994954  2022   340.432313   5846.362364         23  5.830216   

          k         l     wg1_y     wg2_y  ...  (i-wg1)w2g1_k  (i-vg1)w2g1_k  \
0  8.698877  4.897840       NaN  8.806353  ...            NaN           

In [22]:
# Load table working_yearly_g.
# Create summary statistics:
# Take the mean, median, standard deviation, max, min, max (% of median), min (% of median) for each column in the table.
from cvxpy import var
import ibis
from ibis import _, selectors as s
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

g_name = "working_yearly_g"
n_name = "working_yearly_n"


with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    for table_name in [g_name, n_name]:
        # if table_name == n_name:
        #     continue

        this_table = con.table(table_name)

        use_columns = [col for col in this_table.columns if col not in ["registered_number", "year", "pc8", "pc4", "ttwa"]]
        print(use_columns)

        t_summary = (
            this_table
            .select(use_columns)
            .rename({
                "y": "gva",
                "k": "total_assets",
                "l": "employees"
            })
            .aggregate(
                s.across(
                    s.all(),
                    {
                        "mean": _.mean(),
                        "median": _.median(),
                        "std": _.std(),
                        "max": _.max(),
                        "min": _.min(),
                    }
                )
            )
            .pivot_longer(
                s.all(),
                names_to=["variable", "statistic"],
                names_pattern=r"^(.*)_(.*)$",
                values_to="value"
            )
            .pivot_wider(
                names_from="statistic",
                values_from="value"
            )
            .mutate(
                max_percent_mean = ibis.ifelse(_.mean.abs() >= 1, _.max / _.mean - 1, None),
                min_percent_mean = ibis.ifelse(_.mean.abs() >= 1, _.min / _.mean - 1, None),
                # Split the variable name by _ and take the last part as suffix (there may be only one item)
                prefix = _.variable.split("_")[0],
                suffix=_.variable.split("_")[-1]
            )
        )
        # df_summary = (
        #     t_summary    
        #     .order_by("suffix")
        #     .execute()
        # )
        # display(df_summary.style.format({
        #     "mean": "{:,.0f}",
        #     "median": "{:,.0f}",
        #     "std": "{:,.0f}",
        #     "max": "{:,.0f}",
        #     "min": "{:,.0f}",
        #     "max_percent_mean": "{:.1%}",
        #     "min_percent_mean": "{:.1%}"
        # }))

        # Display in dataframe
        # take every t_column, separate by _ the suffix will be the new column of our dataframe
        # the prefix will be our row of our dataframe.
        # For every cell in our dataframe, insert the t_column's mean and standard deviation in that column
        # As a string, joined by (\n), the standard deviation should be in brackets underneath

        t_display = (
            t_summary
            .select("mean", "std", "prefix", "suffix")
            .mutate(
                prefix_mod=_.prefix.re_replace(r"^(?:y|k|l)$", "base"),
            )
            .drop("prefix")
            .pivot_wider(
                names_from="suffix",
                values_from=["mean", "std"]
            )
        )
        df_display = t_display.execute()
        print(df_display.columns)
        # Format mean and std pairs into a combined string per column
        for val in ["y", "k", "l"]:
            mean_col = f"mean_{val}"
            std_col = f"std_{val}"
            if mean_col in df_display.columns and std_col in df_display.columns:
                df_display[val] = [
                    f"{m:,.3f}\n({s:,.3f})" if pd.notnull(m) and pd.notnull(s) else None
                    for m, s in zip(df_display[mean_col], df_display[std_col])
                ]
                df_display = df_display.drop(columns=[mean_col, std_col])
        # Sort the dataframe by prefix_mod, according to the index
        # of which transform it is in the transform_tree.
        sort_key = table_name.split("_")[-1]
        sort_list = [t["out_col"].split("_")[0] for t in transform_tree[sort_key]]
        sort_list_unique = list(dict.fromkeys(sort_list))
        sort_list_unique.insert(0, "base")
        sort_order = {t: i for i, t in enumerate(sort_list_unique)}
        df_sorted = (
            df_display
            .sort_values(by="prefix_mod", key=lambda x: x.map(sort_order), ignore_index=True)
        )
        display(df_sorted)

        # Write as a tab to excel file
        desc_dirs = get_data_dirs(segment="descriptives")
        out_file = desc_dirs.output_dir / f"w_transforms_summ.xlsx"
        df_sorted.to_excel(writer, sheet_name=table_name, index=False)
print(f"Exported summary to {out_file}")

['gva', 'total_assets', 'employees', 'y', 'k', 'l', 'wg1_y', 'wg2_y', 'wg3_y', 'wg1_k', 'wg2_k', 'wg3_k', 'wg1_l', 'wg2_l', 'wg3_l', 'w2g1_k', 'wg2wg1_k', 'wg3wg1_k', 'w2g1_l', 'wg2wg1_l', 'wg3wg1_l', 'wg1wg2_k', 'w2g2_k', 'wg3wg2_k', 'wg1wg2_l', 'w2g2_l', 'wg3wg2_l', 'wg1wg3_k', 'wg2wg3_k', 'w2g3_k', 'wg1wg3_l', 'wg2wg3_l', 'w2g3_l', 'w3g1_k', 'w3g1_l', '(i-wg1)w2g1_k', '(i-vg1)w2g1_k', '(i-wg1)w2g1_l', '(i-vg1)w2g1_l', 'w4g1_k', 'w4g1_l', '(i-wg1)w3g1_k', '(i-vg1)w3g1_k', '(i-wg1)w3g1_l', '(i-vg1)w3g1_l']
Index(['prefix_mod', 'mean_y', 'std_y', 'mean_l', 'std_l', 'mean_k', 'std_k'], dtype='str')


,prefix_mod,y,k,l
0,base,8.475\n(1.580),9.443\n(1.900),4.501\n(1.354)
1,wg1,8.711\n(1.120),9.720\n(1.373),4.634\n(0.960)
2,wg2,8.457\n(0.577),9.422\n(0.714),4.483\n(0.399)
3,wg3,8.462\n(0.293),9.430\n(0.365),4.492\n(0.169)
4,w2g1,NaN,9.720\n(1.346),4.634\n(0.936)
5,wg2wg1,NaN,9.684\n(0.738),4.623\n(0.488)
6,wg3wg1,NaN,9.696\n(0.349),4.634\n(0.199)
7,wg1wg2,NaN,9.513\n(0.664),4.508\n(0.369)
8,w2g2,NaN,9.427\n(0.690),4.488\n(0.380)
9,wg3wg2,NaN,9.409\n(0.373),4.475\n(0.162)


['gva', 'total_assets', 'employees', 'y', 'k', 'l', 'wd1_y', 'wd2_y', 'wd3_y', 'wd1_k', 'wd2_k', 'wd3_k', 'wd1_l', 'wd2_l', 'wd3_l', 'w2d1_k', 'w2d1_l', 'w2d2_k', 'w2d2_l', 'w2d3_k', 'w2d3_l', 'w3d1_k', 'w3d1_l', '(i-wd1)w2d1_k', '(i-wd1)w2d1_l', '(i-vd1)w2d1_k', '(i-vd1)w2d1_l', 'w4d1_k', 'w4d1_l', '(i-wd1)w3d1_k', '(i-wd1)w3d1_l', '(i-vd1)w3d1_k', '(i-vd1)w3d1_l']
Index(['prefix_mod', 'mean_y', 'std_y', 'mean_k', 'std_k', 'mean_l', 'std_l'], dtype='str')


,prefix_mod,y,k,l
0,base,8.475\n(1.580),9.443\n(1.900),4.501\n(1.354)
1,wd1,8.604\n(0.909),9.595\n(1.104),4.588\n(0.752)
2,wd2,8.587\n(1.050),9.573\n(1.278),4.574\n(0.872)
3,wd3,8.548\n(0.600),9.528\n(0.738),4.548\n(0.440)
4,w2d1,NaN,9.638\n(1.016),4.610\n(0.691)
5,w3d1,NaN,9.662\n(0.979),4.624\n(0.668)
6,w4d1,NaN,9.674\n(0.953),4.631\n(0.650)
7,w2d2,NaN,9.604\n(1.227),4.589\n(0.837)
8,w2d3,NaN,9.541\n(0.672),4.553\n(0.389)
9,(i-wd1)w2d1,NaN,-0.024\n(0.627),-0.014\n(0.466)


Exported summary to C:\Users\lazyst\Files\ucl\Dissertation\descriptives\output\w_transforms_summ.xlsx


Anomaly on following row:
registered_number	year	wg1_y	wg2_y	wg3_y	wg1_k	wg2_k	wg3_k	wg1_l	wg2_l	wg3_l	gva1_pc8	gva1_pc4_d	gva1_ttwa_d	total_assets_pc8	total_assets_pc4_d	total_assets_ttwa_d	employees_pc8	employees_pc4_d	employees_ttwa_d	check_y_pc8	check_y_pc4	check_y_ttwa	check_k_pc8	check_k_pc4	check_k_ttwa	check_l_pc8	check_l_pc4	check_l_ttwa	percent_y_pc8	percent_y_pc4	percent_y_ttwa	percent_k_pc8	percent_k_pc4	percent_k_ttwa	percent_l_pc8	percent_l_pc4	percent_l_ttwa
04411560	2009	497.241884	622489.0859	60545.99322	11048.83689	1952732.553	635598.9481	28.66666667	3066.189189	693.7905331	677.4403231	622525.7043	80022.57145	359.5797285	1952751.989	722361.3462	13	3066.891892	820.8799609	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	FALSE	-26.6	-0.01	-24.34	2972.71	0	-12.01	120.51	-0.02	-15.48

In [23]:
# import ibis
# from utils.f_0_dirs import get_data_dirs
# db_path = get_data_dirs().output_dir / "fame_data.duckdb"
# con = ibis.duckdb.connect(db_path)

con.raw_sql("CHECKPOINT;")
con.disconnect()